# BEAR Motor Test — Minimal Controller

Walks one **Koala BEAR Muscle Build (KBMB02)** through every operating mode so you can confirm the actuator is healthy and responds to commands.

**Hardware assumptions** (edit cell 2 if different):
- 1 motor, ID = `1` (factory default)
- USB2BEAR on `/dev/ttyUSB0` at `8 Mbps` (the dangle's fast setting)
- Power: 3–8S Li-Po within 9–33.6 V

**How to use:** run cells top-to-bottom. Stop and run the *EMERGENCY DISABLE* cell any time the motor misbehaves.

**Safety notes baked in:**
- Every mode switch goes *disable → set goals → set mode → enable*. Switching modes while enabled is dangerous — the motor immediately executes whatever `goal_*` is in the new mode's register.
- `limit_i_max`, `limit_velocity_max`, and `limit_position_min/max` are set conservatively before the first enable.
- Final cell disables and closes the port — always run it (or re-run the EMERGENCY DISABLE cell).

## 1. Connect

`Manager.BEAR` is a thin subclass of `Packet.PKT` that opens a `pyserial` port and exposes `get_<reg>` / `set_<reg>` methods for every register in `CONTROL_TABLE.py`.

In [1]:
import time
from pybear import Manager

# Edit these if your setup differs from the defaults.
PORT = '/dev/serial/by-id/usb-Westwood_Robotics_USB2BEAR_Boosted_UB0104-if00-port0'
MOTOR_ID = 1
BAUD = 8000000  # USB2BEAR fast setting

# Constructing the BEAR object opens the serial port immediately.
# single_timeout / bulk_timeout default to 2 ms, single_try_num / bulk_try_num to 2.
bear = Manager.BEAR(
    port=PORT,
    baudrate=BAUD,
    single_timeout=0.01,   # 10 ms — comfortably above FTDI worst case
    single_try_num=3,
    bulk_timeout=0.01,
    bulk_try_num=3,
)

   __        __          _                               _    
   \ \      / /___  ___ | |_ __      __ ___    ___    __| |   
    \ \ /\ / // _ \/ __|| __|\ \ /\ / // _ \  / _ \  / _` |   
     \ V  V /|  __/\__ \| |_  \ V  V /| (_) || (_) || (_| |   
      \_/\_/  \___||___/ \__|  \_/\_/  \___/  \___/  \__,_|   
              ____         _             _    _               
             |  _ \  ___  | |__    ___  | |_ (_)  ___  ___    
             | |_) |/ _ \ | '_ \  / _ \ | __|| | / __|/ __|   
             |  _ <| (_) || |_) || (_) || |_ | || (__ \__ \   
             |_| \_\___/ |_.__/  \___/  \__||_| \___||___/   
                                                              
=== PyBEAR by Westwood Robotics -- Last Updated 2025.07.29 ===



In [2]:
# Diagnostic: find any responding BEAR on this dangle
import serial
from pybear import Manager

# Close any prior bear instance to free the port
try:
    bear.close_port()
except NameError:
    pass

PORT_TRY = '/dev/ttyUSB0'   # or the by-id path

found = []
for baud in (8_000_000, 2_000_000):
    try:
        scan = Manager.BEAR(port=PORT_TRY, baudrate=baud, single_timeout=0.005, single_try_num=1)
    except serial.SerialException as e:
        print(f"baud {baud}: cannot open port — {e}")
        continue
    print(f"\nScanning at {baud} bps...")
    for mid in range(21):
        try:
            data, err = scan.ping(mid)[0]
            if err is not None:
                print(f"  FOUND id={mid} err=0x{err:02x} ({scan.decode_error(err)})")
                found.append((baud, mid, err))
        except Exception as e:
            print(f"  id={mid}: {e!r}")
            break
    scan.close_port()

print("\nResults:", found if found else "Nothing responded on either baud.")


   __        __          _                               _    
   \ \      / /___  ___ | |_ __      __ ___    ___    __| |   
    \ \ /\ / // _ \/ __|| __|\ \ /\ / // _ \  / _ \  / _` |   
     \ V  V /|  __/\__ \| |_  \ V  V /| (_) || (_) || (_| |   
      \_/\_/  \___||___/ \__|  \_/\_/  \___/  \___/  \__,_|   
              ____         _             _    _               
             |  _ \  ___  | |__    ___  | |_ (_)  ___  ___    
             | |_) |/ _ \ | '_ \  / _ \ | __|| | / __|/ __|   
             |  _ <| (_) || |_) || (_) || |_ | || (__ \__ \   
             |_| \_\___/ |_.__/  \___/  \__||_| \___||___/   
                                                              
=== PyBEAR by Westwood Robotics -- Last Updated 2025.07.29 ===


Scanning at 8000000 bps...
[PyBEAR | WARNING] :: Maximum try number reached. Just move on.
[PyBEAR | WARNING] :: Maximum try number reached. Just move on.
[PyBEAR | WARNING] :: Maximum try number reached. Just move on.
  FOUND id=3 err=0x80 (N

## 2. Ping & decode error byte

Every PyBEAR read returns `[([data, ...], error_byte), ...]` — outer list per motor, inner tuple is `(data_list, error_byte)`. The error byte's high bit is always 1, so a healthy reply is `0x80 = 128`. `bear.decode_error()` translates the lower 7 bits to a human-readable string.

In [3]:
# ping returns [([fw_hw_version_int], error_byte)]
ping_data, err = bear.ping(MOTOR_ID)[0]
if err is None:
    raise RuntimeError(f"BEAR {MOTOR_ID} did not respond. Check power, port, baud, and ID.")
print(f"BEAR {MOTOR_ID} online. fw/hw version blob = {ping_data[0]:#x}")
print(f"Error byte = 0x{err:02x} ({bear.decode_error(err)})")

/home/rml/miniforge3/envs/wrbear_test/lib/python3.14/site-packages/pybear/Manager.py:320: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  print("   \ \      / /___  ___ | |_ __      __ ___    ___    __| |   ")
/home/rml/miniforge3/envs/wrbear_test/lib/python3.14/site-packages/pybear/Manager.py:321: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  print("    \ \ /\ / // _ \/ __|| __|\ \ /\ / // _ \  / _ \  / _` |   ")
/home/rml/miniforge3/envs/wrbear_test/lib/python3.14/site-packages/pybear/Manager.py:322: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  print("     \ V  V /|  __/\__ \| |_  \ V  V /| (_) || (_) || (_| |   ")
/home/rml/miniforge3/envs/wrbear_test/lib/python3.14/site-packages/

PortNotOpenError: Attempting to use a port that is not open

## Full error & warning diagnostic (read-only — run any time, e.g. when the red LED is on)

Per the SDK manual, the Red LED lights for an **existing Error**. The 8-bit error code returned with
every packet packs 7 status bits (little-endian):

| bit | type | name |
|---|---|---|
| 0 | Warning | Communication |
| 1 | Warning | Overheat |
| 2 | Error | Absolute Position |
| 3 | Error | Watchdog Timeout & E-STOP |
| 4 | Error | Joint Limit |
| 5 | Error | Hardware Fault |
| 6 | Error | Initialization Error |
| 7 | — | always 1 (marker bit) |

Warnings (bits 0–1) self-clear once the underlying condition goes away. Errors (bits 2–6) **latch** —
`torque_enable = 0` (disable) clears them, but only if the underlying condition is also gone.

This cell only reads registers — it never enables or moves the motor, so it's safe to run while the
red LED is lit or the motor is in E-STOP/damping. It checks the live per-packet error byte, the latched
`error_status`/`warning_status` registers, and cross-checks the live voltage/temperature/position against
their configured limits so you can see *why* a bit is set, not just *that* it is.

In [ ]:
def _val(getter, motor_id):
    """Flatten [([val], err)] -> (val, err) for a single-motor single-reg getter."""
    data_list, err = getter(motor_id)[0]
    return data_list[0], err

# bit -> (name, kind, what it means / how to clear it)
ERROR_BITS = {
    0: ('Communication',             'Warning', 'Corrupted packet received. Self-clears; reflects only the last comms round.'),
    1: ('Overheat',                  'Warning', 'A temp sensor (IC/powerstage/winding) is above temp_limit_low. Self-clears once it cools back below that.'),
    2: ('Absolute Position',         'Error',   'Encoder read error. On Koala: power-cycle and move the shaft, should self-clear; contact WR if it persists. On battery-backed encoders (Panda/Mountain/Kodiak): use set_posi().'),
    3: ('Watchdog Timeout & E-STOP', 'Error',   'Either the ESTOP terminal is floating/not pulled low, 0x03 was written to torque_enable, or (mode 0 only) watchdog_timeout comms deadline was missed.'),
    4: ('Joint Limit',               'Error',   'present_position went outside limit_position_min/max (modes 2/3 only). Disable, bring the shaft back within bounds (or widen the limits).'),
    5: ('Hardware Fault',            'Error',   'input_voltage is outside min_voltage/max_voltage, or a MOSFET driver fault. Disable and fix the supply voltage.'),
    6: ('Initialization Error',      'Error',   'Corrupted flash save file — needs recalibration.'),
}

def _decode_bits(raw):
    if raw is None:
        return []
    raw = int(raw)
    return [(b, *ERROR_BITS[b]) for b in ERROR_BITS if (raw >> b) & 1]

print(f"=== BEAR {MOTOR_ID} full error/warning diagnostic ===\n")

# 1. Is it even answering?
ping_data, perr = bear.ping(MOTOR_ID)[0]
if perr is None:
    raise RuntimeError(f"BEAR {MOTOR_ID} did not respond to ping — check power, cabling, port, baud, and ID before going further.")
print(f"Ping OK. fw/hw = {ping_data[0]:#x}. Live packet error byte = 0x{perr:02x} ({bear.decode_error(perr)})")

# 2. torque_enable tells you disabled / enabled / critical-latched / damping
torque_en, _ = _val(bear.get_torque_enable, MOTOR_ID)
TE_MEANING = {
    0: 'disabled',
    1: 'enabled',
    2: 'DISABLED — critical error latched, motor refuses to enable',
    3: 'SAFE DAMPING — non-critical error tripped while enabled (or E-STOP)',
}
print(f"torque_enable = {torque_en} ({TE_MEANING.get(torque_en, 'unknown')})")

# 3. Latched error_status / warning_status registers (u32 bitmasks — not the same
#    thing as the per-packet error byte above; these persist until cleared).
err_data, _  = bear.single_read(MOTOR_ID, ['error_status'])
warn_data, _ = bear.single_read(MOTOR_ID, ['warning_status'])
err_status, warn_status = err_data[0], warn_data[0]
print(f"\nerror_status   raw = 0b{int(err_status):07b}")
print(f"warning_status raw = 0b{int(warn_status):07b}")

active, seen = [], set()
for bit, name, kind, note in _decode_bits(err_status) + _decode_bits(warn_status):
    if bit not in seen:
        seen.add(bit)
        active.append((bit, name, kind, note))

if not active:
    print("\nNo bits set in error_status or warning_status right now.")
    print("If the red LED is still lit, it may be latched from a transient that hasn't been cleared —")
    print("run the EMERGENCY DISABLE cell (torque_enable=0 clears latching errors).")
else:
    print(f"\n{len(active)} bit(s) active:")
    for bit, name, kind, note in sorted(active):
        print(f"  bit {bit} [{kind:7s}] {name}")
        print(f"      -> {note}")

# 4. Cross-check the live values that actually drive bits 1, 4, 5 so you see *why*.
print("\n--- Live values vs. their config thresholds ---")
v_bus, _ = _val(bear.get_input_voltage, MOTOR_ID)
v_min, _ = _val(bear.get_min_voltage, MOTOR_ID)
v_max, _ = _val(bear.get_max_voltage, MOTOR_ID)
v_ok = v_min <= v_bus <= v_max
print(f"input_voltage      {v_bus:6.2f} V   vs  min/max_voltage      [{v_min:.2f}, {v_max:.2f}] V   "
      f"{'ok' if v_ok else '*** OUT OF RANGE -> Hardware Fault ***'}")

t_wind, _  = _val(bear.get_winding_temperature, MOTOR_ID)
t_power, _ = _val(bear.get_powerstage_temperature, MOTOR_ID)
t_ic, _    = _val(bear.get_ic_temperature, MOTOR_ID)
t_low, _   = _val(bear.get_temp_limit_low, MOTOR_ID)
t_high, _  = _val(bear.get_temp_limit_high, MOTOR_ID)
t_max = max(t_wind, t_power, t_ic)
print(f"max(temperature)  {t_max:6.1f} C   vs  temp_limit_low/high  [{t_low:.1f}, {t_high:.1f}] C   "
      f"{'ok' if t_max < t_low else '*** AT/OVER LIMIT -> Overheat, Iq being throttled ***'}"
      f"  (winding {t_wind:.1f} / powerstage {t_power:.1f} / ic {t_ic:.1f})")

pos_now, _ = _val(bear.get_present_position, MOTOR_ID)
p_min, _   = _val(bear.get_limit_position_min, MOTOR_ID)
p_max, _   = _val(bear.get_limit_position_max, MOTOR_ID)
p_ok = p_min <= pos_now <= p_max
print(f"present_position  {pos_now:6.3f} rad vs  limit_position_min/max [{p_min:.3f}, {p_max:.3f}] rad   "
      f"{'ok' if p_ok else '*** OUT OF RANGE -> Joint Limit ***'}")

wd_us, _ = _val(bear.get_watchdog_timeout, MOTOR_ID)
mode_now, _ = _val(bear.get_mode, MOTOR_ID)
print(f"watchdog_timeout   {wd_us:.0f} us  (only enforced in mode 0/Torque; current mode = {mode_now})")

print("\nReminder: disabling (torque_enable=0) clears LATCHING errors (bits 2-6), but only if the")
print("underlying condition above is also resolved — otherwise it will just latch again on enable.")
print("Bit 3 (E-STOP) additionally needs the physical ESTOP terminal pulled to GND before it will clear.")

## Set / change a motor's ID (one-time setup)

ID is a **Configuration Register**, so unlike the gains in section 4 it *must* be persisted with `save_config` — otherwise it reverts to the last-saved value on the next power cycle. Two rules from the SDK manual:

1. **Disable torque first.** `save_config` writes flash, and the motor can stop responding mid-write if it's enabled.
2. **IDs must be unique on a chain.** Fresh motors all default to ID 1, so renumber **one motor at a time**: connect a single motor, give it its final ID, then add the next. After `set_id` the motor only answers to the *new* ID, so `save_config` is addressed by the new ID.

*(Note: the PDF's CAUTION that `save_config` takes only one ID is stale — the installed PyBEAR 0.1.3 accepts several, e.g. `save_config(2, 3)`. You still renumber one motor at a time because of rule 2.)*

In [13]:
# --- Renumber ONE BEAR: change its ID and persist it to flash ---
# Connect a single un-numbered motor at a time. Fresh motors all default to ID 1,
# so two of them on the same bus collide — give each its final ID before adding the next.
CURRENT_ID = 1   # the ID the motor responds to right now
NEW_ID     = 2   # the ID you want it to have

# 1. Confirm this motor is actually present at its current ID.
if bear.ping(CURRENT_ID)[0][1] is None:
    raise RuntimeError(f"No BEAR at ID {CURRENT_ID}. Is it powered and the only motor on the bus?")

# 2. Disable before writing flash — save_config on an enabled motor can hang it (manual WARNING).
bear.set_torque_enable((CURRENT_ID, 0))

# 3. Change the ID, then persist. After set_id the motor only answers to NEW_ID,
#    so save_config must be addressed by NEW_ID.
bear.set_id((CURRENT_ID, NEW_ID))
bear.save_config(NEW_ID)
time.sleep(1)  # let the flash write finish before we re-ping

# 4. Verify the new ID answers. (If the rename hadn't stuck, the motor would still be
#    on CURRENT_ID and this ping would fail — save_config sends no reply to confirm on its own.)
if bear.ping(NEW_ID)[0][1] is None:
    raise RuntimeError(f"Renumber failed — BEAR {NEW_ID} did not respond after save.")
print(f"Done. Motor is now ID {NEW_ID}. Set MOTOR_ID = {NEW_ID} in cell 2 to control it.")

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Done. Motor is now ID 2. Set MOTOR_ID = 2 in cell 2 to control it.


In [14]:
# Update Motor ID variable
MOTOR_ID = NEW_ID

## 3. Health snapshot

Read the things that determine whether it's safe to enable: bus voltage, three temperatures, and the current shaft position (so we can use it as the initial position goal and avoid jumps).

In [15]:
def read1(getter, motor_id):
    """Convenience: flatten [([val], err)] -> (val, err) for single-motor single-reg reads."""
    data_list, err = getter(motor_id)[0]
    return data_list[0], err

v_bus, _   = read1(bear.get_input_voltage, MOTOR_ID)
t_wind, _  = read1(bear.get_winding_temperature, MOTOR_ID)
t_power, _ = read1(bear.get_powerstage_temperature, MOTOR_ID)
t_ic, _    = read1(bear.get_ic_temperature, MOTOR_ID)
pos0, _    = read1(bear.get_present_position, MOTOR_ID)
torque_en, _ = read1(bear.get_torque_enable, MOTOR_ID)

print(f"Bus voltage      : {v_bus:.2f} V        (KBMB02 wants 9–33.6 V)")
print(f"Winding temp     : {t_wind:.1f} °C")
print(f"Powerstage temp  : {t_power:.1f} °C")
print(f"IC temp          : {t_ic:.1f} °C")
print(f"Present position : {pos0:.4f} rad")
print(f"torque_enable    : {torque_en}  (0 disabled, 1 enabled, 2 critical err, 3 e-stop/damping)")

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Bus voltage      : 12.50 V        (KBMB02 wants 9–33.6 V)
Winding temp     : 20.8 °C
Powerstage temp  : 21.3 °C
IC temp          : 23.4 °C
Present position : 0.9822 rad
torque_enable    : 0  (0 disabled, 1 enabled, 2 critical err, 3 e-stop/damping)


## 4. One-time configuration (motor must be disabled)

Sets the inner current-loop PID (recommended values for KBMB02 from the SDK manual Table 6) and safety limits. We do **not** call `save_config` — these stay in volatile memory and reset on power cycle. That's the safer default while testing.

In [16]:
import math

# Make absolutely sure the motor is off before touching gains/limits.
bear.set_torque_enable((MOTOR_ID, 0))

# --- Current loop (id and iq) — recommended for Koala Muscle Build V2 ---
bear.set_p_gain_id((MOTOR_ID, 0.358))
bear.set_i_gain_id((MOTOR_ID, 0.045))
bear.set_d_gain_id((MOTOR_ID, 0.0))
bear.set_p_gain_iq((MOTOR_ID, 0.358))
bear.set_i_gain_iq((MOTOR_ID, 0.045))
bear.set_d_gain_iq((MOTOR_ID, 0.0))

# --- Safety limits ---
# limit_i_max caps both id and iq. KBMB02 KT = 1.16 Nm/A, so 1.5 A ≈ 1.74 Nm at the output.
bear.set_limit_i_max((MOTOR_ID, 1.5))
# limit_velocity_max applies in Velocity and Position modes (NOT Direct Force).
bear.set_limit_velocity_max((MOTOR_ID, 5.0))
# Position bounds in Position and Direct Force modes — going outside triggers internal damping.
bear.set_limit_position_min((MOTOR_ID, -3*math.pi))
bear.set_limit_position_max((MOTOR_ID,  3*math.pi))

print("Configured. Motor still disabled.")

Configured. Motor still disabled.


## 5. Mode-switch helper

The safe sequence for every mode change: **disable → set goal_xxx for the new mode → set mode → enable**. Skipping the disable step risks an instant jump because the new mode's existing `goal_*` register fires the moment `mode` is written.

In [ ]:
MODE_TORQUE   = 0
MODE_VELOCITY = 1
MODE_POSITION = 2
MODE_FORCE    = 3   # "Direct Force" — combined position/velocity/torque

# PyBEAR writes (including set_mode, set_torque_enable) are fire-and-forget: no ack, no
# retry, unlike reads. On a spotty link a set_mode() can silently drop, leaving the motor
# in the OLD mode while goal_iq is driven as if the new mode had taken effect — if the old
# mode was Position/Direct Force, its limit_position_min/max stays enforced, which is how
# a "Torque mode" run can trip a Joint Limit error that mode 0 should never trigger. So
# switch_mode/disable read back (reads DO retry/verify) and resend until confirmed.
VERIFY_TRIES = 5
VERIFY_DELAY = 0.01

def disable():
    """Idempotent. Safe to call any time. Confirms torque_enable reads back 0 before
    returning — clears latched errors (e.g. Joint Limit), and a dropped disable write
    can't silently leave the latch un-cleared."""
    for attempt in range(VERIFY_TRIES):
        bear.set_torque_enable((MOTOR_ID, 0))
        state, err = read1(bear.get_torque_enable, MOTOR_ID)
        if err is not None and state == 0:
            return
        print(f"  [disable] not confirmed yet (state={state}), retrying ({attempt + 1}/{VERIFY_TRIES})...")
        time.sleep(VERIFY_DELAY)
    raise RuntimeError(f"Disable never confirmed after {VERIFY_TRIES} tries — check the link, or power-cycle the motor.")

def switch_mode(motor_id, new_mode, goals: dict):
    """Disable, prime goals for the new mode, switch mode, re-enable.

    `goals` keys can be any of: 'goal_position', 'goal_velocity', 'goal_iq', 'goal_id'.
    Set whichever ones the new mode actually consumes (others are harmless to skip).
    """
    disable()
    time.sleep(0.01)  # let the motor coast / damping kick in before we rewrite goals
    setters = {
        'goal_position': bear.set_goal_position,
        'goal_velocity': bear.set_goal_velocity,
        'goal_iq':       bear.set_goal_iq,
        'goal_id':       bear.set_goal_id,
    }
    for name, value in goals.items():
        setters[name]((motor_id, value))

    for attempt in range(VERIFY_TRIES):
        bear.set_mode((motor_id, new_mode))
        actual_mode, err = read1(bear.get_mode, motor_id)
        if err is not None and actual_mode == new_mode:
            break
        print(f"  [switch_mode] mode not confirmed yet (read {actual_mode}), retrying ({attempt + 1}/{VERIFY_TRIES})...")
        time.sleep(VERIFY_DELAY)
    else:
        raise RuntimeError(f"set_mode({new_mode}) never confirmed after {VERIFY_TRIES} tries — check the link before enabling.")

    bear.set_torque_enable((motor_id, 1))

print("Helpers defined.")

## 6. Position mode (mode 2)

Position mode cascades: **position PID → velocity PID → iq PID → motor**. We capture the current shaft angle and use it as the initial goal so the motor *holds* rather than snapping. Then we step 0.5 rad and read back.

In [ ]:
# Tune position-loop gains. These work for many KBMB02 setups; reduce if oscillation appears.
bear.set_p_gain_position((MOTOR_ID, 5.0))
bear.set_i_gain_position((MOTOR_ID, 0.0))
bear.set_d_gain_position((MOTOR_ID, 0.2))

# Velocity loop gets used by position mode internally — must also be sane.
bear.set_p_gain_velocity((MOTOR_ID, 0.5))
bear.set_i_gain_velocity((MOTOR_ID, 0.0))
bear.set_d_gain_velocity((MOTOR_ID, 0.0))

# Read present position WHILE DISABLED, then use it as the initial goal so we don't jump.
pos_now, _ = read1(bear.get_present_position, MOTOR_ID)
print(f"Current pos = {pos_now:.4f} rad — using as initial goal.")

switch_mode(MOTOR_ID, MODE_POSITION, {'goal_position': pos_now})
print("Enabled in Position mode, holding current angle.")

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Current pos = 0.9822 rad — using as initial goal.
Enabled in Position mode, holding current angle.


In [9]:
# Read present position WHILE DISABLED, then use it as the initial goal so we don't jump.
pos_now, _ = read1(bear.get_present_position, MOTOR_ID)
print(f"Current pos = {pos_now:.4f} rad — using as initial goal.")

switch_mode(MOTOR_ID, MODE_POSITION, {'goal_position': pos_now})
print("Enabled in Position mode, holding current angle.")

# Step +0.5 rad smoothly over 1 second. Splitting the step into 50 pieces avoids
# slamming the position-PID with a discontinuity.
TARGET_DELTA = -math.pi  # rad
TARGET_POS = 0
TO_POS_DELTA = TARGET_POS - pos_now
STEPS = 200

for i in range(STEPS):
    #goal = pos_now + TARGET_DELTA * (i + 1) / STEPS
    goal = pos_now + TO_POS_DELTA * (i + 1) / STEPS
    bear.set_goal_position((MOTOR_ID, goal))
    time.sleep(0.02)

time.sleep(0.3)  # let it settle
pos_final, err = read1(bear.get_present_position, MOTOR_ID)
#print(f"Goal      : {pos_now + TARGET_DELTA:.4f} rad")
print(f"Goal      : {TARGET_POS:.4f} rad")
print(f"Reached   : {pos_final:.4f} rad")
print(f"Error byte: 0x{err:02x} ({bear.decode_error(err)})")

Current pos = -0.0817 rad — using as initial goal.
Enabled in Position mode, holding current angle.
Goal      : 0.0000 rad
Reached   : -0.0818 rad
Error byte: 0x80 (No Error!)


## 7. Velocity mode (mode 1)

Direct velocity setpoint into the velocity PID. We start at 0, ramp to a slow speed, hold for a moment, ramp back to 0, then disable. Speed is in **rad/s at the output shaft** (after the 20:1 reduction).

In [20]:
switch_mode(MOTOR_ID, MODE_VELOCITY, {'goal_velocity': 0.0})
print("Enabled in Velocity mode at 0 rad/s.")

TARGET_VEL = -2.0   # rad/s — gentle
RAMP_STEPS = 30

# Ramp up
for i in range(RAMP_STEPS):
    bear.set_goal_velocity((MOTOR_ID, TARGET_VEL * (i + 1) / RAMP_STEPS))
    time.sleep(0.02)

# Hold for 1 s while sampling
vel_samples = []
for _ in range(10):
    v, _ = read1(bear.get_present_velocity, MOTOR_ID)
    vel_samples.append(v)
    time.sleep(0.1)
print(f"Mean held velocity: {sum(vel_samples)/len(vel_samples):+.3f} rad/s (target {TARGET_VEL:+.3f})")

# Ramp down
for i in range(RAMP_STEPS):
    bear.set_goal_velocity((MOTOR_ID, TARGET_VEL * (RAMP_STEPS - i - 1) / RAMP_STEPS))
    time.sleep(0.02)

bear.set_goal_velocity((MOTOR_ID, 0.0))
time.sleep(0.2)
disable()
print("Velocity test complete, motor disabled.")

Enabled in Velocity mode at 0 rad/s.
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Mean held velocity: -0.007 rad/s (target -2.000)
Velocity test complete, motor disabled.


## 8. Torque mode (mode 0)

Direct iq command. **Most likely to launch the rotor** if mishandled — there's no closed loop on velocity or position holding it back. We use 0.3 A (~0.35 Nm at the output) and only run it for 0.5 s before zeroing.

If the motor is unloaded, it WILL accelerate. Hold the output shaft or clamp the motor body before running this cell.

In [23]:
TEST_IQ = 0.3  # A. Stays well under limit_i_max = 1.5 A.
BURST_S = 0.5

switch_mode(MOTOR_ID, MODE_TORQUE, {'goal_iq': 0.0})
print(f"Enabled in Torque mode. Applying iq = {TEST_IQ} A for {BURST_S} s — hold the shaft.")

bear.set_goal_iq((MOTOR_ID, TEST_IQ))
t0 = time.time()
iq_samples = []
while time.time() - t0 < BURST_S:
    iq, _ = read1(bear.get_present_iq, MOTOR_ID)
    iq_samples.append(iq)
    time.sleep(0.05)

# Zero torque BEFORE disabling — disabling alone is also safe (it cuts current),
# but zeroing first avoids a tiny disable transient if you choose to keep it enabled.
bear.set_goal_iq((MOTOR_ID, 0.0))
time.sleep(0.1)
disable()

print(f"Mean measured iq: {sum(iq_samples)/len(iq_samples):+.3f} A (commanded {TEST_IQ:+.3f} A)")
print("Torque test complete, motor disabled.")

Enabled in Torque mode. Applying iq = 0.3 A for 0.5 s — hold the shaft.
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Mean measured iq: +0.252 A (commanded +0.300 A)
Torque test complete, motor disabled.


## 9. Direct Force mode (mode 3)

Behaves like a virtual spring-damper around `goal_position`: position error × `p_gain_force` plus velocity error × `d_gain_force`, summed with `goal_iq` feed-forward. Try wiggling the shaft by hand — it should pull back.

**Watch out:** in this mode `limit_velocity_max` is ignored. Keep `limit_i_max` sane (it still applies).

In [25]:
# Spring-damper gains. P = stiffness, D = damping.
bear.set_p_gain_force((MOTOR_ID, 5.0))
bear.set_i_gain_force((MOTOR_ID, 0.0))   # I gain almost always causes oscillation here.
bear.set_d_gain_force((MOTOR_ID, 0.2))

# Anchor to current position; no feed-forward velocity or torque.
pos_now, _ = read1(bear.get_present_position, MOTOR_ID)
switch_mode(MOTOR_ID, MODE_FORCE, {
    'goal_position': pos_now,
    'goal_velocity': 0.0,
    'goal_iq':       0.0,
})
print(f"Direct Force enabled, anchored at {pos_now:.4f} rad. Try moving the shaft.")
print("Sampling for 5 s...")

t0 = time.time()
while time.time() - t0 < 5.0:
    p, _  = read1(bear.get_present_position, MOTOR_ID)
    iq, _ = read1(bear.get_present_iq, MOTOR_ID)
    print(f"  t={time.time()-t0:4.1f}s  pos={p:+.4f} rad  iq={iq:+.3f} A", end='\r')
    time.sleep(0.1)
print()

disable()
print("Direct Force test complete, motor disabled.")

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Direct Force enabled, anchored at 2.5202 rad. Try moving the shaft.
Sampling for 5 s...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single r

## EMERGENCY DISABLE

Run this cell at any time to cut current. Safe to run repeatedly.

In [26]:
try:
    bear.set_torque_enable((MOTOR_ID, 0))
    print("Disabled.")
except Exception as e:
    print(f"Disable attempt failed: {e!r}. Pull power if motor is still moving.")

Disabled.


## 10. Cleanup

Disable and close the serial port. Run this when you're done so the next session can reopen the port without `Resource busy`.

In [8]:
bear.set_torque_enable((MOTOR_ID, 0))
bear.close_port()
print("Disabled and port closed.")

Disabled and port closed.
